# Projeto Completo NDVI - CRISP-DM

Notebook principal do projeto, organizado em CRISP-DM e alinhado ao objetivo descrito em `info.md`.

Estrutura adotada:
- Business Understanding
- Data Understanding
- Data Preparation
- Statistical Analysis
- Modeling
- Evaluation
- Export

Ambientes suportados:
- localmente;
- no Jupyter tradicional;
- no Google Colab com pasta no Drive.


## Uso no Colab

1. Coloque o repositorio no Drive, se a execucao for feita a partir dele.
2. Para nao versionar caminhos locais, use um arquivo `.monolithfarm.paths.json`.
3. Selecione o perfil com `MONOLITHFARM_PROFILE`.
4. Para definir o modo explicitamente, use `MONOLITHFARM_NOTEBOOK_MODE=colab`.
5. Rode a primeira celula para montar o Drive e localizar o projeto.
6. Variaveis opcionais:
   - `MONOLITHFARM_PROJECT_DIR`
   - `MONOLITHFARM_DATA_DIR`
   - `MONOLITHFARM_OUTPUT_DIR`


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path


OUTPUT_SUBDIR = 'notebook_outputs/complete_ndvi'
NOTEBOOK_MODE = os.environ.get("MONOLITHFARM_NOTEBOOK_MODE", "auto").strip().lower()
if NOTEBOOK_MODE not in {"auto", "jupyter", "colab"}:
    raise ValueError("MONOLITHFARM_NOTEBOOK_MODE deve ser `auto`, `jupyter` ou `colab`.")

PROFILE_NAME = os.environ.get("MONOLITHFARM_PROFILE", "").strip()
CONFIG_ENV_PATH = os.environ.get("MONOLITHFARM_PATHS_FILE", "").strip()
IN_COLAB_RUNTIME = "google.colab" in sys.modules
USE_COLAB_MODE = NOTEBOOK_MODE == "colab" or (NOTEBOOK_MODE == "auto" and IN_COLAB_RUNTIME)
REQUIRED_PACKAGES = ['duckdb', 'pandas', 'plotly', 'pyproj', 'shapely', 'scipy']
COLAB_PROJECT_HINTS = [
    "/content/drive/MyDrive/MonolithFarm",
    "/content/drive/My Drive/MonolithFarm",
    "/content/Projeto-FarmLab",
    "/content/MonolithFarm",
]
COLAB_DATA_HINTS = [f"{project_dir}/data" for project_dir in COLAB_PROJECT_HINTS]


def package_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def first_existing_path(candidates: list[str]) -> Path | None:
    for candidate in candidates:
        path = Path(candidate).expanduser()
        if path.exists():
            return path.resolve()
    return None


def discover_paths_config_file() -> Path | None:
    candidates: list[Path] = []
    if CONFIG_ENV_PATH:
        candidates.append(Path(CONFIG_ENV_PATH).expanduser())

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        candidates.append(candidate / ".monolithfarm.paths.json")
        candidates.append(candidate / "monolithfarm.paths.json")

    seen: set[str] = set()
    for candidate in candidates:
        resolved = candidate.resolve() if candidate.is_absolute() else candidate
        key = str(resolved)
        if key in seen:
            continue
        seen.add(key)
        if resolved.exists():
            return resolved
    return None


def load_paths_config() -> tuple[dict, Path | None]:
    config_path = discover_paths_config_file()
    if config_path is None:
        return {}, None

    payload = json.loads(config_path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise ValueError(f"Arquivo de configuracao invalido: {config_path}")
    return payload, config_path


PATHS_CONFIG, PATHS_CONFIG_FILE = load_paths_config()
CONFIG_BASE_DIR = PATHS_CONFIG_FILE.parent.resolve() if PATHS_CONFIG_FILE is not None else None
DEFAULT_PROFILE = str(PATHS_CONFIG.get("default_profile") or "").strip()
ACTIVE_PROFILE = PROFILE_NAME or DEFAULT_PROFILE
PROFILE_SETTINGS = {}
if ACTIVE_PROFILE:
    profiles = PATHS_CONFIG.get("profiles", {})
    if ACTIVE_PROFILE not in profiles:
        raise KeyError(
            f"Perfil `{ACTIVE_PROFILE}` nao encontrado em {PATHS_CONFIG_FILE}."
        )
    profile_value = profiles.get(ACTIVE_PROFILE)
    if isinstance(profile_value, dict):
        PROFILE_SETTINGS = profile_value
elif isinstance(PATHS_CONFIG.get("profile"), dict):
    PROFILE_SETTINGS = PATHS_CONFIG["profile"]


def config_value(key: str):
    return PROFILE_SETTINGS.get(key) if isinstance(PROFILE_SETTINGS, dict) else None


def resolve_config_path(raw_value: object, *, base_dir: Path | None) -> Path | None:
    if raw_value in {None, ""}:
        return None
    path = Path(str(raw_value)).expanduser()
    if not path.is_absolute() and base_dir is not None:
        path = (base_dir / path).resolve()
    else:
        path = path.resolve()
    return path


def mount_colab_drive_if_needed() -> None:
    if not USE_COLAB_MODE:
        return
    if not IN_COLAB_RUNTIME:
        raise RuntimeError(
            "MONOLITHFARM_NOTEBOOK_MODE='colab' foi definido, mas o runtime atual nao e Google Colab."
        )
    from google.colab import drive

    drive_root = Path("/content/drive")
    drive_root.mkdir(parents=True, exist_ok=True)
    if not (drive_root / "MyDrive").exists() and not (drive_root / "My Drive").exists():
        drive.mount("/content/drive")


def find_project_dir() -> Path:
    explicit = os.environ.get("MONOLITHFARM_PROJECT_DIR")
    if explicit:
        return Path(explicit).expanduser().resolve()

    config_project_dir = resolve_config_path(config_value("project_dir"), base_dir=CONFIG_BASE_DIR)
    if config_project_dir is not None:
        return config_project_dir

    if USE_COLAB_MODE:
        hinted_project = first_existing_path(COLAB_PROJECT_HINTS)
        if hinted_project is not None and (hinted_project / "pyproject.toml").exists():
            return hinted_project
        hinted_data = first_existing_path(COLAB_DATA_HINTS)
        if hinted_data is not None and (hinted_data.parent / "pyproject.toml").exists():
            return hinted_data.parent.resolve()

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "Nao foi possivel localizar `pyproject.toml`. Defina MONOLITHFARM_PROJECT_DIR, "
        "MONOLITHFARM_PROFILE ou crie `.monolithfarm.paths.json`."
    )


def find_data_dir(project_dir: Path) -> Path:
    explicit = os.environ.get("MONOLITHFARM_DATA_DIR")
    if explicit:
        return Path(explicit).expanduser().resolve()

    config_data_dir = resolve_config_path(config_value("data_dir"), base_dir=CONFIG_BASE_DIR)
    if config_data_dir is not None:
        return config_data_dir

    if USE_COLAB_MODE:
        hinted_data = first_existing_path(COLAB_DATA_HINTS)
        if hinted_data is not None:
            return hinted_data

    for candidate in [project_dir / "data", project_dir / "FarmLab"]:
        if candidate.exists():
            return candidate.resolve()

    return (project_dir / "data").resolve()


def find_output_dir(project_dir: Path) -> Path:
    explicit = os.environ.get("MONOLITHFARM_OUTPUT_DIR")
    if explicit:
        return Path(explicit).expanduser().resolve()

    config_output_dir = resolve_config_path(config_value("output_dir"), base_dir=CONFIG_BASE_DIR)
    if config_output_dir is not None:
        return config_output_dir

    config_output_root = resolve_config_path(config_value("output_root"), base_dir=CONFIG_BASE_DIR)
    if config_output_root is not None:
        return (config_output_root / OUTPUT_SUBDIR).resolve()

    return (project_dir / OUTPUT_SUBDIR).resolve()


def auto_install_enabled() -> bool:
    explicit = os.environ.get("MONOLITHFARM_AUTO_INSTALL")
    if explicit is not None:
        return explicit == "1"

    config_auto_install = config_value("auto_install")
    if config_auto_install is not None:
        return bool(config_auto_install)

    return USE_COLAB_MODE


mount_colab_drive_if_needed()
PROJECT_DIR = find_project_dir()
DATA_DIR = find_data_dir(PROJECT_DIR)
OUTPUT_DIR = find_output_dir(PROJECT_DIR)
AUTO_INSTALL = auto_install_enabled()

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

if AUTO_INSTALL and any(not package_available(name) for name in REQUIRED_PACKAGES):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_DIR)])
    except Exception:
        subprocess.check_call(["uv", "pip", "install", "--python", sys.executable, "-e", str(PROJECT_DIR)])

print("NOTEBOOK_MODE =", NOTEBOOK_MODE)
print("IN_COLAB_RUNTIME =", IN_COLAB_RUNTIME)
print("USE_COLAB_MODE =", USE_COLAB_MODE)
print("PROFILE_NAME =", PROFILE_NAME or "<default>")
print("PATHS_CONFIG_FILE =", PATHS_CONFIG_FILE)
print("ACTIVE_PROFILE =", ACTIVE_PROFILE or "<none>")
print("PROJECT_DIR =", PROJECT_DIR)
print("DATA_DIR    =", DATA_DIR)
print("OUTPUT_DIR  =", OUTPUT_DIR)
print("AUTO_INSTALL =", AUTO_INSTALL)

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Diretorio de dados nao encontrado: {DATA_DIR}")


In [ ]:
import base64
import math

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display

from farmlab.complete_analysis import build_complete_ndvi_workspace, save_complete_ndvi_outputs


workspace = build_complete_ndvi_workspace(DATA_DIR)
dataset_overview = workspace["dataset_overview"]
numeric_profiles = workspace["numeric_profiles"]
ndvi_stats_by_area = workspace["ndvi_stats_by_area"]
ndvi_outliers = workspace["ndvi_outliers"]
pair_weekly_gaps = workspace["pair_weekly_gaps"]
pair_classic_tests = workspace["pair_classic_tests"]
ndvi_trend_tests = workspace["ndvi_trend_tests"]
weekly_correlations = workspace["weekly_correlations"]

data_audit = workspace["data_audit"]
pair_effect_tests = workspace["pair_effect_tests"]
event_driver_lift = workspace["event_driver_lift"]
transition_model_frame = workspace["transition_model_frame"]
transition_model_summary = workspace["transition_model_summary"]
transition_model_coefficients = workspace["transition_model_coefficients"]
transition_model_predictions = workspace["transition_model_predictions"]
final_hypothesis_register = workspace["final_hypothesis_register"]
decision_summary = workspace["decision_summary"]
ndvi_phase_timeline = workspace["ndvi_phase_timeline"]
ndvi_events = workspace["ndvi_events"]
ndvi_clean = workspace["ndvi_clean"]
weather_daily = workspace["weather_daily"]
ops_area_daily = workspace["ops_area_daily"]
miip_daily = workspace["miip_daily"]
soil_context = workspace["soil_context"]

print("Workspace carregado.")


In [ ]:
def sample_rows_per_area(frame: pd.DataFrame, max_images_per_area: int = 2) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    groups = []
    for _, group in frame.sort_values("date").groupby("season_id", sort=False):
        if len(group) <= max_images_per_area:
            groups.append(group)
            continue
        positions = sorted({0, len(group) // 2, len(group) - 1})
        groups.append(group.iloc[positions[:max_images_per_area]])
    return pd.concat(groups, ignore_index=True)


def _image_to_base64(path: str) -> str | None:
    file_path = Path(path)
    if not file_path.exists():
        return None
    return base64.b64encode(file_path.read_bytes()).decode("ascii")


def gallery_html(frame: pd.DataFrame, *, title: str, subtitle_columns: list[str]) -> HTML:
    if frame.empty:
        return HTML(f"<h2>{title}</h2><p>Sem imagens disponiveis.</p>")

    cards = [f"<h2>{title}</h2>"]
    for row in frame.itertuples(index=False):
        image_b64 = _image_to_base64(str(row.image_path)) if pd.notna(row.image_path) else None
        subtitle = " | ".join(
            f"{column}={getattr(row, column)}" for column in subtitle_columns if hasattr(row, column)
        )
        cards.append(f"<h3>{row.area_label} | {pd.Timestamp(row.date).date()}</h3>")
        cards.append(f"<p>{subtitle}</p>")
        if image_b64 is None:
            cards.append("<p>Imagem nao encontrada.</p>")
            continue
        cards.append(
            f'<img src="data:image/jpeg;base64,{image_b64}" '
            'style="max-width: 560px; border-radius: 8px; margin-bottom: 16px;" />'
        )
    return HTML("\n".join(cards))


def event_gallery_html(events: pd.DataFrame, ndvi_clean: pd.DataFrame, max_events: int = 10) -> HTML:
    if events.empty or ndvi_clean.empty:
        return HTML("<h2>Galeria de eventos</h2><p>Sem eventos ou imagens disponiveis.</p>")

    priority = {"queda_forte": 0, "queda": 1, "baixo_vigor": 2, "recuperacao": 3, "pico": 4}
    selected = events.copy()
    selected["priority"] = selected["event_type"].map(priority).fillna(9)
    selected = selected.sort_values(["priority", "week_start", "area_label"]).head(max_events)

    cards = ["<h2>Eventos NDVI mais relevantes</h2>"]
    for event in selected.itertuples(index=False):
        candidates = ndvi_clean[ndvi_clean["season_id"] == event.season_id].copy()
        if candidates.empty:
            continue
        candidates["date"] = pd.to_datetime(candidates["date"], errors="coerce")
        event_date = pd.to_datetime(event.week_start, errors="coerce")
        match = candidates.loc[(candidates["date"] - event_date).abs().idxmin()]
        image_b64 = _image_to_base64(str(match["image_path"])) if pd.notna(match["image_path"]) else None
        cards.append(f"<h3>{event.area_label} | {pd.Timestamp(event.week_start).date()} | {event.event_type}</h3>")
        cards.append(f"<p><strong>Drivers:</strong> {event.drivers_summary}</p>")
        cards.append(f"<p>{event.story_sentence}</p>")
        if image_b64 is not None:
            cards.append(
                f'<img src="data:image/jpeg;base64,{image_b64}" '
                'style="max-width: 580px; border-radius: 8px; margin-bottom: 18px;" />'
            )
    return HTML("\n".join(cards))


## 1. Business Understanding

O objetivo desta analise e explicar, com base estatistica e agronomica, por que em alguns momentos a area convencional supera a area com tecnologias 4.0 e em outros momentos ocorre o inverso.

O material da aula de ciclo de vida do dado foi usado como referencia metodologica. O notebook segue a estrutura CRISP-DM com foco em problema, dados, preparo, modelagem e avaliacao.


In [ ]:
info_path = PROJECT_DIR / "info.md"
if info_path.exists():
    display(Markdown(info_path.read_text(encoding="utf-8")))
else:
    print("info.md nao encontrado em", info_path)


## 2. Data Understanding

Primeiro validamos o inventario das fontes, o tamanho de cada base e a cobertura temporal.


In [ ]:
raw_frames = {
    "ndvi_clean": ndvi_clean,
    "weather_daily": weather_daily,
    "ops_area_daily": ops_area_daily,
    "miip_daily": miip_daily,
    "soil_context": soil_context,
    "ndvi_phase_timeline": ndvi_phase_timeline,
    "transition_model_frame": transition_model_frame,
}

shape_rows = []
for name, frame in raw_frames.items():
    shape_rows.append(
        {
            "dataset": name,
            "shape": frame.shape,
            "columns": ", ".join(frame.columns[:10]),
        }
    )
pd.DataFrame(shape_rows)


In [ ]:
dataset_overview


In [ ]:
describe_targets = {
    "ndvi_clean": ndvi_clean[["ndvi_mean", "ndvi_std", "soil_pct", "dense_veg_pct", "ndvi_delta", "ndvi_auc"]],
    "weather_daily": weather_daily[
        [
            "precipitation_mm",
            "evapotranspiration_mm",
            "water_balance_mm",
            "temp_avg_c",
            "humidity_avg_pct",
            "wind_avg_kmh",
        ]
    ],
    "ops_area_daily": ops_area_daily[
        [
            column
            for column in [
                "harvest_yield_mean_kg_ha",
                "fert_dose_gap_abs_mean_kg_ha",
                "overlap_area_pct_bbox",
                "stop_duration_h_per_bbox_ha",
            ]
            if column in ops_area_daily.columns
        ]
    ],
    "miip_daily": miip_daily[
        [
            column
            for column in [
                "avg_pest_count",
                "total_pest_count",
                "alert_hits",
                "control_hits",
                "damage_hits",
            ]
            if column in miip_daily.columns
        ]
    ],
}

for name, frame in describe_targets.items():
    display(Markdown(f"### `.describe()` de `{name}`"))
    if frame.empty:
        print("Base vazia.")
        continue
    display(frame.describe().T)


In [ ]:
fig = px.bar(
    data_audit,
    x="area_label",
    y=["ndvi_valid_ratio", "weather_coverage_ratio", "miip_coverage_ratio"],
    barmode="group",
    title="Cobertura por area: NDVI valido, clima e MIIP",
)
fig.update_layout(yaxis_title="Razao de cobertura", legend_title="")
fig


## 3. Data Preparation

Nesta etapa os dados foram integrados por area e por semana, que e a granularidade mais defensavel para comparar dinamica de NDVI, risco operacional, clima e pragas.


In [ ]:
numeric_profiles


In [ ]:
transition_model_frame.head(20)


## 4. Exploratory Statistical Analysis

Aqui entram estatistica descritiva, identificacao de outliers por z-score, trajetoria temporal e tendencia por area.


In [ ]:
ndvi_stats_by_area


In [ ]:
ndvi_trend_tests


In [ ]:
fig = px.line(
    ndvi_phase_timeline.sort_values("week_start"),
    x="week_start",
    y="ndvi_mean_week",
    color="area_label",
    facet_row="comparison_pair",
    markers=True,
    title="NDVI medio semanal por area e por par",
)
fig.update_layout(height=850)
fig


In [ ]:
flagged = ndvi_outliers.copy()
flagged["date"] = pd.to_datetime(flagged["date"], errors="coerce")

fig = px.scatter(
    flagged,
    x="date",
    y="ndvi_zscore",
    color="area_label",
    symbol="outlier_flag",
    facet_row="comparison_pair",
    hover_data=["ndvi_mean", "outlier_direction"],
    title="Z-score do NDVI por cena",
)
fig.add_hline(y=2.0, line_dash="dash", line_color="#b91c1c")
fig.add_hline(y=-2.0, line_dash="dash", line_color="#b91c1c")
fig.update_layout(height=780)
fig


In [ ]:
ndvi_outliers[ndvi_outliers['outlier_flag']].head(30)


## 5. Pairwise Statistical Testing

Esta secao combina os testes pareados ja existentes no projeto com testes classicos adicionais.

Perguntas desta etapa:
- existe diferenca estatisticamente relevante entre 4.0 e convencional?
- essa diferenca aparece no nivel medio do NDVI, na area sob a curva e nos sinais de problema?
- qual teste e mais apropriado em cada caso, dado o comportamento dos gaps semanais?


In [ ]:
pair_effect_tests


In [ ]:
pair_classic_tests


In [ ]:
ndvi_gap_plot = pair_weekly_gaps.copy()
fig = px.line(
    ndvi_gap_plot.dropna(subset=["gap_ndvi_mean_week_4_0_minus_convencional"]),
    x="week_start",
    y="gap_ndvi_mean_week_4_0_minus_convencional",
    color="comparison_pair",
    markers=True,
    title="Gap semanal de NDVI medio: 4.0 - convencional",
    labels={"gap_ndvi_mean_week_4_0_minus_convencional": "Gap NDVI"},
)
fig.add_hline(y=0, line_dash="dash", line_color="#94a3b8")
fig


In [ ]:
frame = pair_classic_tests.copy()
fig = px.bar(
    frame,
    x="metric_label",
    y="mean_favorable_gap_4_0",
    color="favors",
    facet_row="comparison_pair",
    hover_data=["recommended_test", "recommended_p_value", "paired_effect_size_dz"],
    title="Gap medio favoravel ao 4.0 por metrica",
    color_discrete_map={
        "favorece_4_0": "#0f766e",
        "favorece_convencional": "#b91c1c",
        "inconclusivo": "#64748b",
    },
)
fig.update_layout(height=900)
fig


## 6. Correlation Analysis

Correlacao nao prova causalidade, mas ajuda a priorizar variaveis que merecem leitura agronomica mais cuidadosa.


In [ ]:
weekly_correlations.head(40)


In [ ]:
corr_plot = weekly_correlations[
    (weekly_correlations["analysis_target"] == "delta_ndvi_seguinte")
    & (weekly_correlations["comparison_pair"] == "geral")
].head(15)

fig = px.bar(
    corr_plot,
    x="strongest_abs_correlation",
    y="feature",
    orientation="h",
    color="direction",
    hover_data=["pearson_r", "pearson_p", "spearman_rho", "spearman_p", "strength"],
    title="Top correlacoes com o delta do NDVI da semana seguinte",
    color_discrete_map={"positiva": "#0f766e", "negativa": "#b91c1c", "sem_relacao": "#64748b"},
)
fig.update_layout(height=720, yaxis={"categoryorder": "total ascending"})
fig


## 7. Deep Dive de Eventos

Esta secao detalha semanas problema, drivers associados e imagens de apoio.


In [ ]:
event_driver_lift


In [ ]:
fig = px.bar(
    event_driver_lift,
    x="driver",
    y="delta_pp",
    color="evidence_level",
    facet_row="comparison_pair",
    title="Drivers sobre-representados nas semanas problema do NDVI",
    color_discrete_map={"alta": "#0f766e", "media": "#c58b00", "baixa": "#64748b"},
)
fig.update_layout(height=760)
fig


In [ ]:
ndvi_events.head(20)


In [ ]:
event_gallery_html(ndvi_events, ndvi_clean, max_events=12)


## 8. Modeling

A modelagem foi mantida interpretavel. O alvo modelado e a variacao do NDVI na semana seguinte.


In [ ]:
transition_model_summary


In [ ]:
transition_model_coefficients.head(20)


In [ ]:
fig = px.bar(
    transition_model_coefficients.head(20),
    x="coefficient",
    y="feature",
    orientation="h",
    color="direction",
    title="Coeficientes padronizados do modelo de transicao do NDVI",
    color_discrete_map={"aumenta_ndvi_futuro": "#0f766e", "pressiona_ndvi_futuro": "#b91c1c"},
)
fig.update_layout(height=760, yaxis={"categoryorder": "total ascending"})
fig


In [ ]:
fig = px.scatter(
    transition_model_predictions,
    x="target_next_ndvi_delta",
    y="loo_predicted_next_ndvi_delta",
    color="area_label",
    facet_row="comparison_pair",
    title="Ajuste leave-one-area-out: delta real vs previsto do NDVI",
    labels={
        "target_next_ndvi_delta": "Delta real da proxima semana",
        "loo_predicted_next_ndvi_delta": "Delta previsto",
    },
)
fig.add_shape(type="line", x0=-0.2, y0=-0.2, x1=0.2, y1=0.2, line={"dash": "dash", "color": "#94a3b8"})
fig.update_layout(height=760)
fig


## 9. Evaluation

Sintese das hipoteses, decisoes e lacunas ainda abertas.


In [ ]:
final_hypothesis_register


In [ ]:
decision_summary


In [ ]:
sample_rows = sample_rows_per_area(ndvi_clean, max_images_per_area=2)
gallery_html(
    sample_rows,
    title="Galeria visual de apoio por area",
    subtitle_columns=["comparison_pair", "treatment", "ndvi_mean", "soil_pct", "dense_veg_pct"],
)


In [ ]:
pd.DataFrame({"gap": workspace["deep_dive_gaps"]})


## 10. Export

Esta celula grava os CSVs analiticos finais em `OUTPUT_DIR`.


In [ ]:
written_files = save_complete_ndvi_outputs(workspace, OUTPUT_DIR)
pd.DataFrame({"written_file": [str(path) for path in written_files]})
